# RSNA Knee — dry run on a subset

| Experiment | Notebook | What this version does |
|---|---|---|
| **EXP-000** | `hlly34/rsna-knee-dryrun` | subset sanity run: 58 gold + 600 studies, EfficientNet-B0 fold 0, 8 epochs |

Checks the whole pipeline on real DICOMs quickly: build the cache for the 58 gold studies + 600 random ones, train one fold, score it against the radiologist labels, and run test inference exactly as the submission notebook does.

Pipeline code lives in the attached dataset `rsna-knee-code` (repo `src/`, `configs/`, `scripts/`); this notebook only orchestrates it. Results are registered with `scripts/track.py` into `experiments/registry.csv` using the id above.

In [ ]:
EXP = {"id": "EXP-000", "notebook": "hlly34/rsna-knee-dryrun", "what": "subset sanity run: 58 gold + 600 studies, EfficientNet-B0 fold 0, 8 epochs"}
print("EXPERIMENT", EXP["id"], "-", EXP["what"])
LIMIT, EPOCHS, CFG = 600, 8, 'effnet_224'
import glob, json, os, subprocess, sys, time

def walk_find(pred, root="/kaggle/input", max_depth=6):
    """Depth-limited search of the attached inputs that never descends into the DICOM trees."""
    root = root.rstrip("/")
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        dirnames[:] = [] if depth >= max_depth else sorted(d for d in dirnames if not d.endswith("_series"))
        hit = pred(dirpath, filenames)
        if hit:
            return hit
    return None

# Kaggle mounts inputs at /kaggle/input/{competitions,datasets,notebooks}/... - locate them instead of hardcoding
CODE = walk_find(lambda d, f: d if os.path.isfile(os.path.join(d, "src", "models", "inference.py")) else None)
COMP = walk_find(lambda d, f: d if "test.csv" in f and os.path.isdir(os.path.join(d, "test_series")) else None)
assert CODE and COMP, (CODE, COMP)
sys.path.insert(0, CODE)
print("code:", CODE, "| competition:", COMP)


## 1. Environment
Packages missing from the Kaggle image are installed (internet is on here). The pylibjpeg wheels are also saved to the output so the offline inference notebook can install them.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "iterative-stratification", "lightning",
                "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg"], check=True)
import torch, timm, pydicom
print("torch", torch.__version__, "| timm", timm.__version__, "| pydicom", pydicom.__version__,
      "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], "| CPUs:", os.cpu_count())
os.makedirs("/kaggle/working/wheels", exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "download", "-q", "-d", "/kaggle/working/wheels",
                "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg"], check=False)

## 2. Labels
Public report-derived label sets are scored against the 58 gold studies, then blended (`configs/default.yaml → labels`).

In [ ]:
DATA = "/kaggle/tmp/data"            # scratch space (not saved); outputs go to /kaggle/working
os.makedirs(f"{DATA}/labels", exist_ok=True)
if not os.path.exists(f"{DATA}/raw"):
    os.symlink(COMP, f"{DATA}/raw")
for name in ["llm_labels_v4_blend.csv", "llm_labels_full.csv", "report_labels_v2.csv"]:
    src = walk_find(lambda d, f: os.path.join(d, name) if name in f else None)
    print(f"{name:28s} <- {src}")
    if src and not os.path.exists(f"{DATA}/labels/{name}"):
        os.symlink(src, f"{DATA}/labels/{name}")
ENV = dict(os.environ, DATA_DIR=DATA, PYTHONPATH=CODE)
os.chdir("/kaggle/working")

def run(*args, env=None, log=None):
    """Run a repo script/module as a subprocess (fresh CUDA context, clean logs)."""
    t = time.time()
    out = open(log, "w") if log else None
    subprocess.run([sys.executable, *args], env=env or ENV, check=True, stdout=out, stderr=subprocess.STDOUT if out else None)
    print(f"[{' '.join(map(str, args[:2]))}] {time.time() - t:.0f}s")

run(f"{CODE}/scripts/eval_labels.py")

## 3. DICOM → cached stacks
For each study: 6 plane/sequence slots, slices sorted by position, knee side from geometry (left knees mirrored), 140 mm crop, 256 px, 72 slices → one `.npz`.

In [ ]:
os.makedirs(f"{DATA}/processed", exist_ok=True)
run(f"{CODE}/scripts/preprocess.py", f"exp={CFG}", f"preprocess.limit={LIMIT}", "preprocess.num_workers=4")
import numpy as np
files = sorted(glob.glob(f"{DATA}/processed/cache_256/*.npz"))
sides = [int(np.load(f)["side"]) for f in files]
print(f"cached {len(files)} | knee R/L/unknown = {sides.count(1)}/{sides.count(-1)}/{sides.count(0)}")

In [ ]:
import matplotlib.pyplot as plt
d = {s: next(f for f, x in zip(files, sides) if x == s) for s in (1, -1) if s in sides}
fig, ax = plt.subplots(len(d), 6, figsize=(18, 3 * len(d)), squeeze=False)
for r, (s, f) in enumerate(d.items()):
    z = np.load(f)
    for c in range(6):
        idx = np.where(z["slot"] == c)[0]
        ax[r, c].imshow(z["image"][idx[len(idx) // 2]], cmap="gray"); ax[r, c].axis("off")
        ax[r, c].set_title(f"{'R' if s == 1 else 'L'} knee slot {c}")
plt.tight_layout(); plt.savefig("/kaggle/working/canonical_examples.png", dpi=60); plt.show()

## 4. Train fold 0 and score against gold

In [ ]:
run("-m", "src.models.train_pl", f"exp={CFG}", "fold=0", f"training.num_epochs={EPOCHS}",
    "resources.num_workers=4", "training.batch_size=16")
print(open(glob.glob("models/*/fold0_metrics.json")[0]).read())

## 5. Test inference (same code path as the submission notebook)

In [ ]:
run("-m", "src.models.inference", "--data_dir", COMP, "--ckpts", "models/**/fold*.pt",
    "--cache_dir", "/kaggle/tmp/test_cache", "--workers", "4", "--out", "/kaggle/working/submission.csv")
json.dump(EXP, open("/kaggle/working/experiment.json", "w"))